In [1]:
from pathlib import Path

from katabatic.artifacts import LocalArtifactStore
from katabatic.models.medgan.models import MEDGAN
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_dataset

ROOT = None

for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "datasets").exists() and (p / "models").exists():
        ROOT = p
        break

raw_file = ROOT / "datasets" / "magic.csv"
processed_file = ROOT / "preprocessed_data" / "magic.csv"
artifact_dir = ROOT / "artifacts"

print("Raw dataset:", raw_file)
print("Dataset exists:", raw_file.exists())

processed_file.parent.mkdir(parents=True, exist_ok=True)

preprocess_dataset(
    str(raw_file),
    str(processed_file),
    target_col="class"
)

store = LocalArtifactStore(str(artifact_dir))

pipeline = TrainTestSplitPipeline(
    model=MEDGAN(
        ae_pretrain_epochs=10,
        gan_epochs=10
    )
)

pipeline._evaluations = []

results = pipeline.run(
    input_csv=str(processed_file),
    dataset_name="magic",
    artifact_store=store,
    model_name="medgan",
)

print(results)

Raw dataset: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\magic.csv
Dataset exists: True
Preprocessing: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\magic.csv
Saved preprocessed dataset to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\preprocessed_data\magic.csv
Loaded data with shape: (19020, 11)


INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Training MedGAN Model
INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Loaded training data: (15216, 10)
INFO:katabatic.models.medgan.models:Categorical columns: ['class']
INFO:katabatic.models.medgan.models:Continuous columns: ['fLength', 'fWidth', 'fSize', 'fConc', 'fConc1', 'fAsym', 'fM3Long', 'fM3Trans', 'fAlpha', 'fDist']
INFO:katabatic.models.medgan.models:Data normalized to [0, 1] range
INFO:katabatic.models.medgan.models:Original range: [0.00, 12.00]
INFO:katabatic.models.medgan.models:Normalized range: [0.00, 1.00]
INFO:katabatic.models.medgan.models:
Phase 1: Pretraining Autoencoder for 10 epochs...


Train label distribution:
 class
g    0.648396
h    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
g    0.648265
h    0.351735
Name: proportion, dtype: float64
Saved dataset artifact under datasets/magic/split-20260808-003622


INFO:katabatic.models.medgan.models:Epoch 1/10: AE Loss = 0.598998
INFO:katabatic.models.medgan.models:Epoch 10/10: AE Loss = 0.511515
INFO:katabatic.models.medgan.models:
Phase 2: Training GAN for 10 epochs...
INFO:katabatic.models.medgan.models:Epoch 1/10: D Loss = 1.300268, G Loss = 0.774232
INFO:katabatic.models.medgan.models:Epoch 10/10: D Loss = 0.227485, G Loss = 10.256484
INFO:katabatic.models.medgan.models:
Generating 15216 synthetic samples...
INFO:katabatic.models.medgan.models:
Synthetic data saved to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\models\medgan_magic_train-20260808-003623\synthetic
INFO:katabatic.models.medgan.models:Training complete!


{'message': 'Train test split pipeline executed successfully.', 'dataset_ref': DatasetRef(dataset_name='magic', dataset_version='split-20260808-003622'), 'model_ref': ModelRef(model_name='medgan', dataset_name='magic', dataset_version='split-20260808-003622', train_run_id='train-20260808-003623'), 'evaluation_refs': []}


In [2]:
import pandas as pd

from katabatic.pipeline.evaluation_pipeline import SyntheticEvaluationPipeline

splits_root = ROOT / "artifacts" / "datasets" / "magic"

split_dirs = sorted(
    [p for p in splits_root.glob("split-*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)

latest_split = split_dirs[-1]

print("Using split:", latest_split)

train_df = pd.read_csv(
    latest_split / "train" / "train_full.csv"
)

test_df = pd.read_csv(
    latest_split / "test" / "test_full.csv"
)

target_col = "class"

categorical_cols = []

continuous_cols = [
    "fLength",
    "fWidth",
    "fSize",
    "fConc",
    "fConc1",
    "fAsym",
    "fM3Long",
    "fM3Trans",
    "fAlpha",
    "fDist",
]

model = pipeline.model

synthetic_df = model.sample(
    len(train_df),
    seed=42
)

print("Synthetic type:", type(synthetic_df))
print("\nSynthetic sample:")
print(synthetic_df.head())

evaluation_pipeline = SyntheticEvaluationPipeline(
    dimensions=[
        "fidelity",
        "utility",
        "diversity",
        "privacy",
        "consistency",
        "stability",
    ],
    categorical_cols=categorical_cols,
    continuous_cols=continuous_cols,
)

report = evaluation_pipeline.run(
    real_data=train_df,
    synthetic_data=synthetic_df,
    target_col=target_col,
    test_data=test_df,
    model=model,
)

print("\nComposite Score:", report.composite_score)

print("\nDimension Scores:")
for dimension, score in report.dimension_scores.items():
    print(f"{dimension}: {score}")

Using split: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\datasets\magic\split-20260808-003622
Synthetic type: <class 'pandas.core.frame.DataFrame'>

Synthetic sample:
    fLength     fWidth     fSize     fConc    fConc1     fAsym   fM3Long  \
0  7.454941  10.081206  2.566774  3.163660  3.653057  1.905156  3.578218   
1  7.240040   9.513907  2.422920  3.133368  3.549669  2.398618  4.246480   
2  6.364328   6.233359  1.693112  3.589294  3.030572  2.653896  7.484006   
3  5.808081   7.481545  1.756903  3.304155  3.483974  2.761074  5.206701   
4  6.459965   6.236445  1.719952  3.624958  3.116745  2.684625  7.352256   

   fM3Trans    fAlpha     fDist class  
0  5.308019  6.013037  2.399020     h  
1  5.434614  5.467517  2.263451     h  
2  4.124672  1.614946  2.335538     g  
3  5.060145  3.406628  1.215953     h  
4  4.279755  1.575034  2.233540     g  

Running fidelity evaluation...

=== Fidelity Evaluation ===
Overall fidelity score: 0.79